# Module 3 — Demo lab 2: five worked EDA examples

Not graded · Same five problems as [Read 1–2](../lectures/module-3-read-01) interpretive examples — with full code.

Prerequisites: [Demo 1 — tools](module-3-lab-demo-tools).

Each problem runs the same investigation cycle you will use in Project 3: Ask a question (about one or two variables) → Plan (dataset, cleaning, which summaries and graph, and why) → Collect (load the data with `read_csv()`) → Analyze (check variable names and types, check missingness, build a complete-case tibble, compute the numerical summaries, and graph).

| Problem | Situation |
|---------|-----------|
| 1 | One numerical (histogram + summaries) |
| 2 | One categorical (bar + counts) |
| 3 | Numerical × categorical (boxplots) |
| 4 | Two numerical (scatter + `cor()`) |
| 5 | Two categorical (two-way table + stacked bar) |


## Setup

```{r}
library(tidyverse)
# install.packages("tidyverse")  # if needed
if (!requireNamespace("palmerpenguins", quietly = TRUE)) {
  install.packages("palmerpenguins", repos = "https://cloud.r-project.org")
}
library(palmerpenguins)
```

Quick guide: The reads show tables and figures to interpret; this demo lab teaches the R code for all five EDA situations (plus Pearson `cor()` for linear scatterplots).


------------------------------------------------------------------------

## Demo 2 — Worked examples

*Legacy Lab 9*

Then try [exercise lab](module-3-lab-exercise) on your Project 3 dataset.

For every problem, practice spotting the parts you will change for your own data: the dataset name and each variable's name and type (check types with `glimpse()`). An "Identify" prompt with the answer follows each code chunk — these are exactly the pieces you will swap in the exercise lab and Project 3.

Each example is laid out as Ask → Plan → Collect → Analyze, and each Collect+Analyze code cell is self-contained (it loads the data itself), so you can copy a whole problem and swap in your own dataset and variables.


### Problem 1 — One numerical variable

Ask: How long are penguin flippers, and how spread out are the lengths? (one numerical variable: `flipper_length_mm`)

Plan:

- Dataset: the palmerpenguins penguins data, loaded from a CSV.
- Variable: `flipper_length_mm` — numerical.
- Cleaning: complete-case — drop rows missing `flipper_length_mm`.
- Summaries: mean, median, SD, and IQR, because a numerical variable is described by its center and spread.
- Graph: a histogram (and a dot plot), because one numerical variable is shown as a distribution — shape, center, spread.


### Code anatomy — Problem 1

| Part | Role |
|------|------|
| <span style="color:#2563eb">read_csv(...)</span> | Collect: load the raw CSV into a tibble |
| <span style="color:#2563eb">glimpse()</span> | Check column names and types |
| <span style="color:#2563eb">sum(is.na(x))</span> | Count missing values |
| <span style="color:#2563eb">drop_na()</span> | Keep complete cases, stored as a new tibble |
| <span style="color:#2563eb">mean / median / sd / IQR (na.rm = TRUE)</span> | Center and spread of a numerical variable |
| <span style="color:#2563eb">geom_histogram(), geom_dotplot()</span> | Distribution of one numerical variable |


In [ ]:
# --- Collect: load the raw data from its CSV ---
penguins_raw <- read_csv(
  "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-07-28/penguins.csv",
  show_col_types = FALSE
)

# --- Analyze ---
# 1. Check variable names and types
glimpse(penguins_raw)

# 2. Check missingness in the variable we use
penguins_raw |> summarize(n_missing = sum(is.na(flipper_length_mm)))

# 3. Build a complete-case tibble (new name; raw data left unchanged)
flippers <- penguins_raw |>
  select(flipper_length_mm) |>
  drop_na()

# 4. Numerical summaries: center and spread
flippers |> summarize(
  n      = n(),
  mean   = mean(flipper_length_mm),
  median = median(flipper_length_mm),
  sd     = sd(flipper_length_mm),
  iqr    = IQR(flipper_length_mm)
)

# 5. Graphs: histogram (and a dot plot) of one numerical variable
ggplot(flippers, aes(x = flipper_length_mm)) +
  geom_histogram(binwidth = 5, boundary = 180, fill = "steelblue", color = "white") +
  labs(title = "Penguin flipper length", x = "Flipper length (mm)", y = "Count")

ggplot(flippers, aes(x = flipper_length_mm)) +
  geom_dotplot(binwidth = 5, method = "histodot", fill = "steelblue") +
  labs(title = "Flipper length (dot plot)", x = "Flipper length (mm)", y = NULL) +
  scale_y_continuous(NULL, breaks = NULL)


Identify before you reuse this: the raw dataset, the complete-case tibble you create, and the variable with its type. Confirm with `glimpse(penguins_raw)`.

Answer: raw `penguins_raw` → complete-case `flippers`; variable `flipper_length_mm` — numerical.

Interpret: <span style="color:#7c3aed">mean, median, sd, IQR</span> give the center and spread, and the histogram shows the shape. Project 3 requires one numeric summary and one graph per variable.


### Problem 2 — One categorical variable

Ask: How many penguins are of each species, and what share of the sample is each? (one categorical variable: `species`)

Plan:

- Dataset: the palmerpenguins penguins data, loaded from a CSV.
- Variable: `species` — categorical.
- Cleaning: complete-case — drop rows missing `species`.
- Summaries: counts and proportions (percent), because a categorical variable is described by how many fall in each level.
- Graph: a bar chart, because one categorical variable is shown as one bar per level.


### Code anatomy — Problem 2

| Part | Role |
|------|------|
| <span style="color:#2563eb">read_csv(...)</span> | Collect: load the raw CSV |
| <span style="color:#2563eb">glimpse()</span> | Check names and types |
| <span style="color:#2563eb">drop_na()</span> | Keep complete cases (new tibble) |
| <span style="color:#2563eb">count(species)</span> | Rows = levels; column `n` = counts |
| <span style="color:#2563eb">mutate(prop = n / sum(n))</span> | Proportions $\hat{p}$ from counts |
| <span style="color:#2563eb">geom_bar()</span> | One bar per category |


In [ ]:
# --- Collect: load the raw data ---
penguins_raw <- read_csv(
  "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-07-28/penguins.csv",
  show_col_types = FALSE
)

# --- Analyze ---
# 1. Check names and types
glimpse(penguins_raw)

# 2. Check missingness
penguins_raw |> summarize(n_missing = sum(is.na(species)))

# 3. Complete-case tibble
species_data <- penguins_raw |>
  select(species) |>
  drop_na()

# 4. Counts and proportions (share of the sample)
species_data |>
  count(species) |>
  mutate(prop = n / sum(n))

# 5. Graph: bar chart of one categorical variable
ggplot(species_data, aes(x = species)) +
  geom_bar(fill = "darkorange") +
  labs(title = "Penguin species", x = "Species", y = "Count")


Identify before you reuse this: the raw dataset, the complete-case tibble, and the variable with its type. Confirm with `glimpse(penguins_raw)`.

Answer: raw `penguins_raw` → complete-case `species_data`; variable `species` — categorical.

Interpret: the count/proportion table gives the size and share of each species, and the bar chart shows the same counts visually.

### Problem 3 — One numerical and one categorical variable

Ask: Does flipper length differ across the three penguin species? (one numerical variable `flipper_length_mm` by one categorical variable `species`)

Plan:

- Dataset: the palmerpenguins penguins data, loaded from a CSV.
- Variables: `species` — categorical (the groups); `flipper_length_mm` — numerical.
- Cleaning: complete-case — drop rows missing either variable.
- Summaries: median and IQR (and mean and SD) per group, because we compare a numerical variable across categories.
- Graph: side-by-side boxplots, because they compare a numerical distribution across groups.


### Code anatomy — Problem 3

| Part | Role |
|------|------|
| <span style="color:#2563eb">read_csv(...)</span> | Collect: load the raw CSV |
| <span style="color:#2563eb">glimpse()</span> | Check names and types |
| <span style="color:#2563eb">across(c(...), ~ sum(is.na(.)))</span> | Count missing values in several columns |
| <span style="color:#2563eb">drop_na()</span> | Keep complete cases (new tibble) |
| <span style="color:#2563eb">group_by() + summarize()</span> | One summary row per group |
| <span style="color:#2563eb">geom_boxplot()</span> | Side-by-side boxplots |
| <span style="color:#dc2626">species, flipper_length_mm</span> | Replace with your grouping and numerical variables |


In [ ]:
# --- Collect ---
penguins_raw <- read_csv(
  "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-07-28/penguins.csv",
  show_col_types = FALSE
)

# --- Analyze ---
# 1. Names and types
glimpse(penguins_raw)

# 2. Missingness in both variables
penguins_raw |>
  summarize(across(c(species, flipper_length_mm), ~ sum(is.na(.))))

# 3. Complete-case tibble
flip_by_species <- penguins_raw |>
  select(species, flipper_length_mm) |>
  drop_na()

# 4. Summaries per group: center and spread
flip_by_species |>
  group_by(species) |>
  summarize(
    n      = n(),
    median = median(flipper_length_mm),
    iqr    = IQR(flipper_length_mm),
    mean   = mean(flipper_length_mm),
    sd     = sd(flipper_length_mm)
  )

# 5. Graph: side-by-side boxplots
ggplot(flip_by_species, aes(x = species, y = flipper_length_mm, fill = species)) +
  geom_boxplot() +
  labs(title = "Flipper length by species", x = "Species", y = "Flipper length (mm)") +
  theme(legend.position = "none")


Identify before you reuse this: the raw dataset, the complete-case tibble, and each variable with its type. Confirm with `glimpse(penguins_raw)`.

Answer: raw `penguins_raw` → complete-case `flip_by_species`; `species` — categorical (grouping); `flipper_length_mm` — numerical.

Interpret: <span style="color:#7c3aed">median, iqr</span> compare center and spread per group, and the boxplots show each species' five-number summary side by side.


### Problem 4 — Two numerical variables

Ask: Is bill length associated with flipper length — and if so, in which direction and how strongly? (two numerical variables `bill_length_mm` and `flipper_length_mm`)

Plan:

- Dataset: the palmerpenguins penguins data, loaded from a CSV.
- Variables: `bill_length_mm` and `flipper_length_mm` — both numerical.
- Cleaning: complete-case — drop rows missing either variable.
- Summary: Pearson correlation `r`, but only if the scatterplot looks roughly linear.
- Graph: a scatterplot, because two numerical variables are shown as points — form, direction, strength.


### Code anatomy — Problem 4

| Part | Role |
|------|------|
| <span style="color:#2563eb">read_csv(...)</span> | Collect: load the raw CSV |
| <span style="color:#2563eb">glimpse()</span> | Check names and types |
| <span style="color:#2563eb">across(c(...), ~ sum(is.na(.)))</span> | Count missing values |
| <span style="color:#2563eb">drop_na()</span> | Keep complete cases (new tibble) |
| <span style="color:#2563eb">geom_point()</span> | Scatterplot — two numerical variables |
| <span style="color:#2563eb">cor(x, y)</span> | Pearson $r$ — linear direction and strength ($-1$ to $1$) |
| <span style="color:#2563eb">aes(..., color = species)</span> | Optional: color points by a categorical variable |


In [ ]:
# --- Collect ---
penguins_raw <- read_csv(
  "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-07-28/penguins.csv",
  show_col_types = FALSE
)

# --- Analyze ---
# 1. Names and types
glimpse(penguins_raw)

# 2. Missingness in both variables
penguins_raw |>
  summarize(across(c(bill_length_mm, flipper_length_mm), ~ sum(is.na(.))))

# 3. Complete-case tibble (keep species too, for optional color)
bill_flip <- penguins_raw |>
  select(bill_length_mm, flipper_length_mm, species) |>
  drop_na()

# 4. Graph first: check the form before trusting a correlation
ggplot(bill_flip, aes(x = bill_length_mm, y = flipper_length_mm, color = species)) +
  geom_point(alpha = 0.7) +
  labs(title = "Bill length vs flipper length", x = "Bill length (mm)", y = "Flipper length (mm)")

# 5. Summary: Pearson r (only meaningful if the scatter looks roughly linear)
bill_flip |> summarize(r = cor(bill_length_mm, flipper_length_mm))


Identify before you reuse this: the raw dataset, the complete-case tibble, and each variable with its type. Confirm with `glimpse(penguins_raw)`.

Answer: raw `penguins_raw` → complete-case `bill_flip`; `bill_length_mm` — numerical (x); `flipper_length_mm` — numerical (y); `species` — categorical (point color).

Interpret: describe the form (linear/curved/none), direction (positive/negative/none), and strength (strong/moderate/weak). Then read <span style="color:#7c3aed">cor()</span>: sign = direction; $|r|$ near 1 = strong linear trend. If the plot is curved, `r` alone is not enough.


### Problem 5 — Two categorical variables

Ask: How are the three species distributed across the islands — does the species mix depend on the island? (two categorical variables `species` and `island`)

Plan:

- Dataset: the palmerpenguins penguins data, loaded from a CSV.
- Variables: `species` and `island` — both categorical.
- Cleaning: complete-case — drop rows missing either variable.
- Summaries: a two-way count table, then proportions. With two categorical variables there are three kinds of proportion, so pick the one that matches the question (see the anatomy below).
- Graph: a 100% stacked bar chart, because it compares the species composition within each island.


### Code anatomy — Problem 5 (three kinds of proportion)

A raw count `n` from `count(island, species)` can be turned into three different proportions. They answer different questions, so choose deliberately:

- Joint proportion — divide each count by the grand total (`n / sum(n)` over the whole table). All cells together sum to 1. Answers: "what share of all penguins are on island X and of species Y?"
- Conditional within island (row proportion) — `group_by(island)` then `mutate(n / sum(n))`. Each island sums to 1. Answers: "within an island, what fraction is each species?"
- Conditional within species (column proportion) — `group_by(species)` then `mutate(n / sum(n))`. Each species sums to 1. Answers: "for a given species, what fraction lives on each island?"

A 100% stacked bar with `x = island, fill = species` displays the conditional-within-island proportions — every bar fills to 100%.

| Part | Role |
|------|------|
| <span style="color:#2563eb">read_csv(...)</span> | Collect: load the raw CSV |
| <span style="color:#2563eb">count(island, species)</span> | Two-way counts |
| <span style="color:#2563eb">pivot_wider()</span> | Long counts to a wide table |
| <span style="color:#2563eb">group_by(...) then mutate(n / sum(n))</span> | Conditional proportions |
| <span style="color:#2563eb">geom_bar(position = "fill")</span> | 100% stacked bar |
| <span style="color:#dc2626">species, island</span> | Replace with your two categorical variables |


In [ ]:
# --- Collect ---
penguins_raw <- read_csv(
  "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-07-28/penguins.csv",
  show_col_types = FALSE
)

# --- Analyze ---
# 1. Names and types
glimpse(penguins_raw)

# 2. Missingness in both variables
penguins_raw |>
  summarize(across(c(species, island), ~ sum(is.na(.))))

# 3. Complete-case tibble
spec_island <- penguins_raw |>
  select(species, island) |>
  drop_na()

# 4a. Two-way count table (wide)
spec_island |>
  count(island, species) |>
  pivot_wider(names_from = species, values_from = n, values_fill = 0)

# 4b. JOINT proportions: each cell as a share of ALL penguins (all cells sum to 1)
spec_island |>
  count(island, species) |>
  mutate(joint = n / sum(n))

# 4c. CONDITIONAL within island (row proportions): each island sums to 1
spec_island |>
  count(island, species) |>
  group_by(island) |>
  mutate(prop_within_island = n / sum(n))

# 4d. CONDITIONAL within species (column proportions): each species sums to 1
spec_island |>
  count(island, species) |>
  group_by(species) |>
  mutate(prop_within_species = n / sum(n))

# 5. Graph: 100% stacked bar = species composition WITHIN each island (matches 4c)
ggplot(spec_island, aes(x = island, fill = species)) +
  geom_bar(position = "fill") +
  labs(title = "Species composition within each island", x = "Island", y = "Proportion") +
  scale_y_continuous(labels = scales::percent)


Identify before you reuse this: the raw dataset, the complete-case tibble, and each variable with its type. Confirm with `glimpse(penguins_raw)`.

Answer: raw `penguins_raw` → complete-case `spec_island`; `species` — categorical; `island` — categorical.

Interpret: match the proportion to the question. The 100% stacked bar shows the conditional-within-island proportions (4c) — read each bar as "within this island, what fraction is each species?" Use joint proportions (4b) for "share of all penguins in this island-and-species combination," and within-species proportions (4d) for "where does each species live?" If the bars differ across islands, the species mix depends on the island.
